In [2]:
import pandas as pd
import time ,re,os, hashlib, inspect, tempfile
import dateutil, base64, pdfplumber, ocrmypdf
from bs4 import BeautifulSoup
from dateutil.parser import parse
from io import StringIO, BytesIO

In [13]:
path = r"C:\Users\kaustubh.keny\Projects\OFFICE PROJECTS\cestat\cestat_gov_2025_all_benches\companies.csv"

df = pd.read_csv(path,index_col=False)

df.iloc[:,1].to_json("ok.json",orient="records")

In [ ]:
import tabula
import pandas as pd


def pdf_to_excel(pdf_file_path, excel_file_path):
    # Read PDF file
    tables = tabula.read_pdf(pdf_file_path, pages='all')

    # Write each table to a separate sheet in the Excel file
    with pd.ExcelWriter(excel_file_path) as writer:
        for i, table in enumerate(tables):
            table.to_excel(writer, sheet_name=f'Sheet{i+1}')


pdf_path =r"C:\Users\kaustubh.keny\Downloads\IFSCA\ifsca-bulletin-april-june-202306102023042858.pdf"
pdf_to_excel(pdf_path, 'ifsca-bulletin-april-june.xlsx')

In [6]:
import requests
from bs4 import BeautifulSoup
import csv
import calendar
import time

session = requests.Session()

# Step 1: Get CSRF token from form page
form_url = "https://cestat.gov.in/order-status"
resp = session.get(form_url, verify=True)
soup = BeautifulSoup(resp.text, "html.parser")
csrf_token = soup.find("input", {"name": "csrf_token"})["value"]

post_url = "https://cestat.gov.in/ajax/order-status-web"

results = []

['124438', '109120', '129525', '104044', '133568', '107079', '136507', '119315', '127482']

# Step 2: Loop through months of 2025
# for month in range(1, 13):
#     from_date = f"01-{month:02d}-2025"
#     last_day = calendar.monthrange(2025, month)[1]
#     to_date = f"{last_day:02d}-{month:02d}-2025"

#     payload = {
#         "tab": 4,
#         "csrf_token": csrf_token,
#         "bench": 127482,
#         "from": from_date,
#         "to": to_date,
#         "captcha_code": "111111",   # safe to leave blank
#         "data_table_length": 200
#     }

#     response = session.post(post_url, data=payload, verify=True)
#     response_json = response.json()

#     # Step 3: Parse JSON rows
#     for row in response_json.get("data", []):
#         serial = row[0]
#         case_no = row[1]
#         parties = row[2] #row[2].replace("<br>", " vs ")
#         date = row[3]

#         # Extract href from HTML
#         soup = BeautifulSoup(row[4], "html.parser")
#         link_tag = soup.find("a")
#         pdf_url = None
#         if link_tag and link_tag.get("href"):
#             href = link_tag["href"].replace("./", "")
#             pdf_url = f"https://cestat.gov.in/{href}"

#         results.append({
#             "month": month,
#             "serial": serial,
#             "case_no": case_no,
#             "parties": parties,
#             "date": date,
#             "pdf_url": pdf_url
#         })
    
#     time.sleep(5)   

from_date = f"01-02-2025"
# last_day = calendar.monthrange(2025, month)[1]
to_date = f"12-02-2025"

payload = {
    "tab": 4,
    "csrf_token": csrf_token,
    "bench": 127482,
    "from": from_date,
    "to": to_date,
    "captcha_code": "111111",   # safe to leave blank
    "data_table_length": 200
}

response = session.post(post_url, data=payload, verify=True)
response_json = response.json()

# Step 3: Parse JSON rows
for row in response_json.get("data", []):
    serial = row[0]
    case_no = row[1]
    parties = row[2].replace("<br>", "") #row[2].replace("<br>", " vs ")
    date = row[3]
    
    p1,p2 = parties.split("vs")

    # Extract href from HTML
    soup = BeautifulSoup(row[4], "html.parser")
    link_tag = soup.find("a")
    pdf_url = None
    if link_tag and link_tag.get("href"):
        href = link_tag["href"].replace("./", "")
        pdf_url = f"https://cestat.gov.in/{href}"

    results.append({
        # "month": month,
        "serial": serial,
        "case_no": case_no,
        "parties": parties,
        "p1":p1.lower().strip(),
        "p2":p2.lower().strip(),
        "date": date,
        "pdf_url": pdf_url
    })

time.sleep(5)   


# Step 4: Save results to CSV
with open(f"cestat_{payload["bench"]}_2025.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["serial", "case_no","parties" ,"p1","p2", "date", "pdf_url"])
    writer.writeheader()
    writer.writerows(results)

# print("Saved results to cestat_cases_4_2025.csv")